# Used Car Price Prediction (India) - Regression Project

**Problem statement:** Given details of a used car (brand, age, kilometres driven, fuel, engine, etc.), predict its selling price in INR.

**Why it matters:** Buyers and sellers (CarDekho, Cars24, OLX Autos) need a fair price. A model trained on past sales can give an instant estimate.

**Dataset:** "Vehicle dataset from CarDekho" on Kaggle, file `Car details v3.csv`
(https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho)

**Type of ML problem:** *Regression* (we predict a continuous number, the price).

**Workflow:** Load -> Clean -> EDA -> Preprocess -> Train 4 models -> Compare -> Tune best -> Explain -> Save model

## 1. Setup and loading the data

**Concept - Libraries:** `pandas` handles tables, `numpy` handles numbers, `matplotlib`/`seaborn` draw charts, `scikit-learn` has the ML tools.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import joblib

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

sns.set_style("whitegrid")
RANDOM_STATE = 42
os.makedirs("images", exist_ok=True)   # charts get saved here for your README/report
print("Libraries ready. XGBoost available:", HAS_XGB)

**Loading the CSV.** Download `Car details v3.csv` from the Kaggle link above and put it in the same folder as this notebook.
(In Google Colab: run the second cell below and pick the file when asked.)

In [ ]:
# --- Only needed in Google Colab: uncomment these 2 lines and run ---
# from google.colab import files
# files.upload()

DATA_PATH = "Car details v3.csv"
df = pd.read_csv(DATA_PATH)
print("Rows, columns:", df.shape)
df.head()

## 2. Understanding the dataset

Each **row** is one car that was sold. Each **column** is a **feature** (an input) except `selling_price`, which is the **target** (what we predict).

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print()
print("Missing values per column:")
print(df.isnull().sum())

**What to notice (write these in your report):**
- `mileage`, `engine`, `max_power` are stored as **text** with units (e.g. "19.7 kmpl", "1248 CC"), so a model can't use them yet. We must convert them to numbers.
- There are some **missing values** and **duplicate rows**.
- `torque` is messy text in many formats. We will drop it.

## 3. Data cleaning and feature engineering

**Concept - Feature engineering:** creating better inputs from raw columns.
- `year` -> **car_age** (age is more meaningful than the year itself).
- `name` -> **brand** (the first word, e.g. "Maruti", "Hyundai").
- Text with units -> numbers.

**Why remove duplicates?** If the same car appears twice, one copy could land in training and the other in testing. The model would "recognise" it and look better than it really is. This is a form of **data leakage**.

In [ ]:
data = df.copy()

# 1) Remove duplicate records
before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(data)} duplicate rows. Remaining: {len(data)}")

# 2) Text with units -> numbers (extract the first number in the text)
for col in ["mileage", "engine", "max_power"]:
    data[col] = data[col].astype(str).str.extract(r"([\d.]+)", expand=False).astype(float)

# 3) Brand from name
data["brand"] = data["name"].str.split().str[0]
data["brand"] = data["brand"].replace({"Land": "Land Rover"})

# 4) Car age from year
CURRENT_YEAR = 2026
data["car_age"] = CURRENT_YEAR - data["year"]

# 5) Drop columns we no longer need
data = data.drop(columns=["name", "year", "torque"], errors="ignore")
data.head()

### Outliers

**Concept - Outlier:** a value far from the rest (e.g. a car with 2.3 million km driven). Outliers can distort models, especially Linear Regression.

We look at the extremes first, then remove clearly unrealistic values.

In [ ]:
print(data[["selling_price", "km_driven"]].describe(percentiles=[0.01, 0.5, 0.99, 0.995]).T)

In [ ]:
n0 = len(data)
price_cap = data["selling_price"].quantile(0.995)
data = data[(data["km_driven"] <= 500_000) & (data["selling_price"] <= price_cap)]
data = data[data["selling_price"] > 10_000].reset_index(drop=True)   # removes absurdly cheap/erroneous prices
print(f"Removed {n0 - len(data)} outlier rows. Remaining: {len(data)}")
print(f"Price cap used (99.5th percentile): Rs {price_cap:,.0f}")

## 4. Exploratory Data Analysis (EDA)

EDA = looking at the data with charts to find patterns **before** modelling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data["selling_price"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Selling price (right-skewed)")
sns.histplot(np.log1p(data["selling_price"]), bins=50, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("log(Selling price) (much more bell-shaped)")
plt.tight_layout(); plt.savefig("images/01_price_distribution.png", dpi=150); plt.show()

**Insight:** Prices are heavily **right-skewed** (many cheap cars, few expensive ones). Taking the logarithm makes the distribution much more symmetric. We will train models on `log(price)` and convert predictions back to rupees. This usually improves accuracy for price problems.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.scatterplot(data=data, x="car_age", y="selling_price", alpha=0.3, ax=axes[0])
axes[0].set_title("Price vs car age")
sns.scatterplot(data=data, x="km_driven", y="selling_price", alpha=0.3, ax=axes[1])
axes[1].set_title("Price vs km driven")
sns.scatterplot(data=data, x="max_power", y="selling_price", alpha=0.3, ax=axes[2])
axes[2].set_title("Price vs max power (bhp)")
plt.tight_layout(); plt.savefig("images/02_numeric_vs_price.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), ["fuel", "transmission", "seller_type", "owner"]):
    order = data.groupby(col)["selling_price"].median().sort_values(ascending=False).index
    sns.boxplot(data=data, x=col, y="selling_price", order=order, ax=ax)
    ax.set_title(f"Price by {col}")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig("images/03_categorical_vs_price.png", dpi=150); plt.show()

In [ ]:
top_brands = data["brand"].value_counts().head(10).index
brand_median = (data[data["brand"].isin(top_brands)]
                .groupby("brand")["selling_price"].median().sort_values())
plt.figure(figsize=(9, 4.5))
brand_median.plot(kind="barh", color="teal")
plt.title("Median selling price - 10 most common brands")
plt.xlabel("Price (Rs)")
plt.tight_layout(); plt.savefig("images/04_brand_median_price.png", dpi=150); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
corr = data.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between numeric features")
plt.tight_layout(); plt.savefig("images/05_correlation_heatmap.png", dpi=150); plt.show()

**Concept - Correlation:** a number from -1 to +1 showing how two columns move together. +1 means they rise together, -1 means one rises as the other falls, 0 means no linear relation.

**Observations to write in your report** (check them against YOUR charts and adjust the wording):
1. Price falls as car age and km driven increase.
2. Price rises strongly with `max_power` and `engine` size (bigger, more powerful cars cost more).
3. Diesel and automatic cars tend to have higher median prices than petrol and manual cars.
4. Premium brands (BMW, Audi, Mercedes, etc.) are far more expensive than mass-market brands.
5. `engine`, `max_power` and `seats` are correlated with each other (multicollinearity). This matters for Linear Regression but tree-based models handle it well.

## 5. Data preprocessing

**Concept - Features (X) and target (y):** X = the inputs, y = the answer we want to predict.

**Concept - Train/test split:** we hide 20% of the data (the *test set*). The model learns from the other 80% (the *training set*) and then we check its score on the hidden 20%. This tells us how it will behave on **new** cars.

**Concept - Preprocessing steps:**
- *Missing values* -> fill numeric gaps with the **median**, categorical gaps with the **most frequent** value.
- *Scaling* -> put numeric columns on a similar scale (needed for Linear Regression; harmless for trees).
- *One-hot encoding* -> turn categories like "Diesel/Petrol" into 0/1 columns, since models only understand numbers.

**Avoiding data leakage:** we put all of this inside a scikit-learn **Pipeline**. Then the medians, scaling values, etc. are learned from the *training data only* and merely applied to the test data.

In [ ]:
TARGET = "selling_price"
X = data.drop(columns=[TARGET])
y = data[TARGET]

num_cols = ["car_age", "km_driven", "mileage", "engine", "max_power", "seats"]
cat_cols = ["brand", "fuel", "seller_type", "transmission", "owner"]
X = X[num_cols + cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Training rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

`handle_unknown="ignore"` means that if the app later receives a brand the model never saw, it won't crash.

**Wrapping with log-target:** `TransformedTargetRegressor` trains on `log(price)` and automatically converts predictions back to rupees.

In [ ]:
def make_model(estimator):
    """Preprocessing + estimator in one pipeline, trained on log(price), predicting in Rs."""
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

## 6. Training the models

All models use the **same split and same preprocessing**, so the comparison is fair.

| Model | Idea in plain English |
|---|---|
| **Linear Regression** | Draws the best straight-line style formula. Simple, fast, our *baseline*. |
| **Random Forest** | Many decision trees vote; averaging reduces mistakes. |
| **Gradient Boosting** | Trees built one after another, each fixing the previous one's errors. |
| **XGBoost** | A faster, more optimised version of gradient boosting (very popular in industry). |

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}
if HAS_XGB:
    models["XGBoost"] = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                                     random_state=RANDOM_STATE, n_jobs=-1)

fitted = {}
for name, est in models.items():
    m = make_model(est)
    m.fit(X_train, y_train)
    fitted[name] = m
    print("Trained:", name)

## 7. Model evaluation and comparison

**Regression metrics:**
- **MAE** (Mean Absolute Error): on average, how many rupees is the prediction off? Easiest to explain.
- **MSE** (Mean Squared Error): average of squared errors; punishes big mistakes heavily.
- **RMSE**: square root of MSE, back in rupees.
- **R2 score**: fraction of price variation the model explains. 1.0 = perfect, 0 = no better than guessing the average.

Metrics like Accuracy, Precision, Recall, F1 and ROC-AUC are for *classification* problems, so they don't apply here. Explain this in your report.

We also run **5-fold cross-validation** on the training data: the training set is split into 5 parts, the model trains on 4 and checks on 1, rotating 5 times. If the CV score is close to the test score, the result is reliable.

In [ ]:
def evaluate(model, name):
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring="r2", n_jobs=-1).mean()
    return {"Model": name,
            "MAE (Rs)": mean_absolute_error(y_test, pred),
            "MSE": mse,
            "RMSE (Rs)": np.sqrt(mse),
            "R2 (test)": r2_score(y_test, pred),
            "R2 (5-fold CV)": cv_r2}

results = pd.DataFrame([evaluate(m, n) for n, m in fitted.items()])
results = results.sort_values("R2 (test)", ascending=False).reset_index(drop=True)
results.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
order = results["Model"]
sns.barplot(data=results, x="Model", y="R2 (test)", order=order, ax=axes[0], palette="viridis")
axes[0].set_title("R2 score (higher is better)"); axes[0].set_ylim(0, 1)
sns.barplot(data=results, x="Model", y="MAE (Rs)", order=order, ax=axes[1], palette="magma")
axes[1].set_title("MAE in Rs (lower is better)")
sns.barplot(data=results, x="Model", y="RMSE (Rs)", order=order, ax=axes[2], palette="rocket")
axes[2].set_title("RMSE in Rs (lower is better)")
for ax in axes: ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig("images/06_model_comparison.png", dpi=150); plt.show()

In [ ]:
best_name = results.loc[0, "Model"]
print("Best baseline model:", best_name)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pred = fitted[best_name].predict(X_test)
axes[0].scatter(y_test, pred, alpha=0.3)
lims = [0, max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "r--"); axes[0].set_xlabel("Actual price"); axes[0].set_ylabel("Predicted price")
axes[0].set_title(f"{best_name}: actual vs predicted (closer to red line = better)")
sns.histplot(y_test - pred, bins=50, ax=axes[1], color="gray")
axes[1].set_title("Prediction errors (residuals)"); axes[1].set_xlabel("Actual - predicted (Rs)")
plt.tight_layout(); plt.savefig("images/07_actual_vs_predicted.png", dpi=150); plt.show()

**Explaining which model is best (fill in with YOUR numbers):**
Linear Regression is the baseline. It usually scores lowest because price depends on the features in a *non-linear* way (e.g. the first few years of age drop price faster than later years). The tree-based models (Random Forest, Gradient Boosting, XGBoost) capture these non-linear patterns and usually reach a noticeably higher R2 and lower MAE. State the winner, quote its R2/MAE/RMSE from the table, and mention that CV and test scores agree.

## 8. Hyperparameter tuning

**Concept - Hyperparameters:** settings you choose *before* training (e.g. how many trees, how deep). Default values are rarely optimal.

**RandomizedSearchCV** tries a number of random combinations, scores each using cross-validation, and keeps the best. It is much faster than trying every combination (GridSearchCV).

We tune the best-performing tree model and compare **before vs after**.

In [ ]:
param_spaces = {
    "Random Forest": {
        "regressor__model__n_estimators": [200, 400, 600],
        "regressor__model__max_depth": [None, 10, 20, 30],
        "regressor__model__min_samples_split": [2, 5, 10],
        "regressor__model__min_samples_leaf": [1, 2, 4],
        "regressor__model__max_features": ["sqrt", 0.5, 1.0],
    },
    "Gradient Boosting": {
        "regressor__model__n_estimators": [200, 400, 600],
        "regressor__model__learning_rate": [0.03, 0.05, 0.1],
        "regressor__model__max_depth": [3, 4, 5, 6],
        "regressor__model__subsample": [0.7, 0.85, 1.0],
        "regressor__model__min_samples_leaf": [1, 3, 5],
    },
    "XGBoost": {
        "regressor__model__n_estimators": [300, 500, 800],
        "regressor__model__learning_rate": [0.03, 0.05, 0.1],
        "regressor__model__max_depth": [4, 6, 8],
        "regressor__model__subsample": [0.7, 0.85, 1.0],
        "regressor__model__colsample_bytree": [0.6, 0.8, 1.0],
    },
}

# Tune the best *tree-based* model (Linear Regression has nothing worth tuning here)
tree_results = results[results["Model"].isin(param_spaces)]
tune_name = tree_results.iloc[0]["Model"]
print("Tuning:", tune_name)

search = RandomizedSearchCV(
    make_model(models[tune_name]),
    param_distributions=param_spaces[tune_name],
    n_iter=15, cv=3, scoring="r2",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
search.fit(X_train, y_train)
print("\nBest parameters:")
for k, v in search.best_params_.items():
    print(f"  {k.replace('regressor__model__', '')}: {v}")

In [ ]:
tuned_model = search.best_estimator_
tuned_name = f"{tune_name} (tuned)"
comparison = pd.DataFrame([evaluate(fitted[tune_name], f"{tune_name} (before)"),
                           evaluate(tuned_model, tuned_name)])
comparison.round(3)

In [ ]:
plt.figure(figsize=(7, 4))
comparison_melt = comparison.melt(id_vars="Model", value_vars=["R2 (test)"])
sns.barplot(data=comparison_melt, x="Model", y="value", palette="crest")
plt.ylim(max(0, comparison["R2 (test)"].min() - 0.05), 1)
plt.title("R2 before vs after tuning"); plt.ylabel("R2 (test)")
plt.tight_layout(); plt.savefig("images/08_tuning_before_after.png", dpi=150); plt.show()

**Interpretation:** compare the two rows. If R2 went up and MAE/RMSE went down, tuning helped. If the change is tiny, say so honestly: it means the default model was already close to its limit for this data. Either answer is fine as long as you explain it.

## 9. Which features matter most? (Explainability)

**Permutation importance:** shuffle one column at a time and see how much the score drops. A big drop means the model relied heavily on that column. It works with any model.

In [ ]:
final_model = tuned_model if comparison.loc[1, "R2 (test)"] >= comparison.loc[0, "R2 (test)"] else fitted[tune_name]
final_name = tuned_name if final_model is tuned_model else tune_name

perm = permutation_importance(final_model, X_test, y_test, n_repeats=5,
                              random_state=RANDOM_STATE, scoring="r2", n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
plt.figure(figsize=(8, 5))
imp.plot(kind="barh", color="steelblue")
plt.title(f"Feature importance - {final_name}")
plt.xlabel("Drop in R2 when the feature is shuffled")
plt.tight_layout(); plt.savefig("images/09_feature_importance.png", dpi=150); plt.show()
imp.sort_values(ascending=False).round(4)

**Business insight to write:** name the top 3 features from your chart and explain them in plain words (e.g. "car age and max power are the biggest drivers of price, so sellers of newer, more powerful cars can expect a premium").

## 10. Save the final model (for the Streamlit app)

We save the whole pipeline (preprocessing + model) in one file, plus the list of dropdown options for the app.

In [ ]:
print("Final model:", final_name)
final_scores = evaluate(final_model, final_name)
print({k: (round(v, 3) if not isinstance(v, str) else v) for k, v in final_scores.items()})

joblib.dump(final_model, "car_price_model.joblib", compress=3)

meta = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "brands": sorted(X["brand"].dropna().unique().tolist()),
    "fuels": sorted(X["fuel"].dropna().unique().tolist()),
    "seller_types": sorted(X["seller_type"].dropna().unique().tolist()),
    "transmissions": sorted(X["transmission"].dropna().unique().tolist()),
    "owners": sorted(X["owner"].dropna().unique().tolist()),
    "medians": {c: float(X[c].median()) for c in num_cols},
    "model_name": final_name,
    "r2": float(final_scores["R2 (test)"]),
    "mae": float(final_scores["MAE (Rs)"]),
}
with open("model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

results.round(3).to_csv("model_comparison_results.csv", index=False)
comparison.round(3).to_csv("tuning_before_after.csv", index=False)
print("Saved: car_price_model.joblib, model_meta.json, model_comparison_results.csv, tuning_before_after.csv")

In [ ]:
# Quick sanity check: predict for one example car
example = pd.DataFrame([{
    "car_age": 5, "km_driven": 50000, "mileage": 20.0, "engine": 1197, "max_power": 82.0, "seats": 5,
    "brand": "Maruti", "fuel": "Petrol", "seller_type": "Individual",
    "transmission": "Manual", "owner": "First Owner",
}])
print(f"Estimated price: Rs {final_model.predict(example)[0]:,.0f}")

## 11. Conclusion and future scope (edit with your own results)

**Conclusion:** We built an end-to-end regression pipeline on the CarDekho used-car dataset. After cleaning, feature engineering and log-transforming the target, tree-based ensembles clearly outperformed the Linear Regression baseline. The final model reached R2 = ___ with a Mean Absolute Error of about Rs ___ on unseen cars.

**Limitations:** the data is from a single platform and period; it has no information on car condition, accident history, city, or colour; prices change over time.

**Future scope:** add city and condition features; try stacking/voting ensembles; use more recent listings; add a price-range (prediction interval) instead of a single number; monitor the model for price drift.